# Bab 10 — Integrasi balancing dengan model

Memerlukan output/ dari notebook 01. Kode 10.6 dan 10.7 memakan beberapa menit pada mesin dua inti (hybrid paling lambat). Prediksi tersimpan di output/prediksi/ dan dipakai Bab 11.

## Kode 10.1 (kode101)

In [ ]:
import os, json, time, platform, warnings
from collections import Counter
import numpy as np, pandas as pd
import sklearn, imblearn, xgboost
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline            # pipeline yang menerima sampler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
os.makedirs("output/prediksi", exist_ok=True)

# Data dari Bab 8: X_train/X_test asli (tanpa balancing), label multikelas dan biner
X_train = pd.read_csv("output/X_train.csv");  X_test = pd.read_csv("output/X_test.csv")
y_train = pd.read_csv("output/y_train.csv")["y"].to_numpy()
y_test = pd.read_csv("output/y_test.csv")["y"].to_numpy()
yb_train = pd.read_csv("output/y_train_biner.csv")["y_biner"].to_numpy()
yb_test = pd.read_csv("output/y_test_biner.csv")["y_biner"].to_numpy()
nama_kelas = pd.read_csv("output/pemetaan_label.csv")["label"].tolist()

kolom_num = X_train.select_dtypes(include="number").columns.tolist()
kolom_kat = [c for c in X_train.columns if c not in kolom_num]
print("Latih:", X_train.shape, "| Uji:", X_test.shape)
print("Fitur numerik:", len(kolom_num), "| kategorikal (tidak dipakai di bab ini):", kolom_kat)

def praproses():
    """Imputasi median dan scaling untuk fitur numerik. Dipasang di dalam pipeline,
    sehingga statistiknya hanya dipelajari dari data latih (atau lipatan latih)."""
    return ColumnTransformer(
        [("num", SkPipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())]), kolom_num)],
        remainder="drop")                      # kolom kategorikal dibuang (subbab 10.1)

## Kode 10.2 (kode102)

In [ ]:
# Demonstrasi pada data sintetis kecil: sklearn Pipeline menolak sampler, imblearn Pipeline menerimanya
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

Xs, ys = make_classification(n_samples=500, weights=[0.9, 0.1], random_state=0)
try:
    SkPipeline([("sampler", SMOTE(random_state=0)), ("model", LogisticRegression())]).fit(Xs, ys)
except TypeError as e:
    print("sklearn Pipeline ->", type(e).__name__, ":", str(e).split("'")[0].strip())

pipa = Pipeline([("sampler", SMOTE(random_state=0)), ("model", LogisticRegression())])
pipa.fit(Xs, ys)                                   # sampler aktif saat fit
print("Label sebelum sampler:", {int(k): v for k, v in Counter(ys).items()}, "| setelah sampler (dilihat model):",
      {int(k): v for k, v in Counter(SMOTE(random_state=0).fit_resample(Xs, ys)[1]).items()})
print("predict pada 7 baris ->", pipa.predict(Xs[:7]).shape, "(sampler dilewati)")

## Kode 10.3 (kode103)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def model_rf(class_weight=None):
    return RandomForestClassifier(n_estimators=100, max_depth=None,
                                  class_weight=class_weight,
                                  n_jobs=-1, random_state=RANDOM_STATE)

p = model_rf().get_params()
print("Random Forest:", p["n_estimators"], "pohon | max_depth =", p["max_depth"], "| class_weight =", p["class_weight"])

## Kode 10.4 (kode104)

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

def model_xgb():
    return XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                         tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE)

# Fungsi tujuan dipilih otomatis dari jumlah kelas pada label (diuji pada data sintetis)
for k in (2, 12):
    Xd, yd = make_classification(n_samples=1500, n_informative=8, n_classes=k, random_state=0)
    m = model_xgb(); m.set_params(n_estimators=5).fit(Xd, yd)
    print(f"{k} kelas -> objective =", m.get_xgb_params()["objective"])

# Pembanding tanpa sampling: bobot sampel seimbang (XGBoost) dan class_weight (Random Forest)
bobot = compute_sample_weight("balanced", y_train)
print("Bobot sampel multikelas: terkecil", bobot.min().round(3), "| terbesar", bobot.max().round(1))

## Kode 10.5 (kode105)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

recall_kelas = []          # recall per kelas tiap konfigurasi (dipakai Bab 11)

def gmean(y_true, y_pred, labels):
    """Rata-rata geometrik recall tiap kelas. Bernilai 0 bila ada kelas yang tidak terdeteksi."""
    r = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return float(np.prod(r) ** (1 / len(r)))

def evaluasi(skenario, model_nama, sampler_nama, pipa, X_tr, y_tr, X_te, y_te, fit_kwargs=None):
    """Latih pipeline pada data latih, nilai pada data uji ASLI; catat metrik dan waktu."""
    labels = np.unique(y_te)
    t0 = time.perf_counter(); pipa.fit(X_tr, y_tr, **(fit_kwargs or {}))
    waktu_fit = time.perf_counter() - t0                   # termasuk praproses dan sampling
    tp = []
    for _ in range(5):                                      # ulangi agar waktu prediksi stabil
        t0 = time.perf_counter(); y_pred = pipa.predict(X_te); tp.append(time.perf_counter() - t0)
    proba = pipa.predict_proba(X_te).astype("float32")
    np.savez_compressed(f"output/prediksi/{skenario}_{model_nama}_{sampler_nama}.npz".replace(" ", "_"),
                        y_pred=y_pred, proba=proba)
    for k, r in zip(labels, recall_score(y_te, y_pred, labels=labels, average=None, zero_division=0)):
        recall_kelas.append({"skenario": skenario, "model": model_nama, "sampler": sampler_nama,
                             "kelas": int(k), "recall": float(r)})
    return {"skenario": skenario, "model": model_nama, "sampler": sampler_nama, "status": "ok",
            "akurasi": accuracy_score(y_te, y_pred),
            "presisi_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
            "gmean": gmean(y_te, y_pred, labels),
            "waktu_fit_dtk": waktu_fit, "waktu_prediksi_dtk": float(np.median(tp))}

## Kode 10.6 (kode106)

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN

def konfigurasi(target_rus="auto", target_ros="auto", K=5):
    """Satu kamus nama -> sampler. Semua sampler memakai RANDOM_STATE yang sama."""
    r = RANDOM_STATE
    smote = lambda: SMOTE(sampling_strategy=target_ros, k_neighbors=K, random_state=r)
    return {
        "Baseline": None,
        "RUS": RandomUnderSampler(sampling_strategy=target_rus, random_state=r),
        "ROS": RandomOverSampler(sampling_strategy=target_ros, random_state=r),
        "SMOTE": smote(),
        "Borderline-SMOTE": BorderlineSMOTE(sampling_strategy=target_ros, k_neighbors=K, random_state=r),
        "ADASYN": ADASYN(sampling_strategy=target_ros, n_neighbors=K, random_state=r),
        "SMOTE-Tomek": SMOTETomek(smote=smote(), random_state=r),
        "SMOTE-ENN": SMOTEENN(smote=smote(), random_state=r),
    }

def jalankan_semua(skenario, y_tr, y_te, sampler_dict, hasil):
    """Seluruh kombinasi model x sampler; konfigurasi sama kecuali sampler."""
    for model_nama, buat in (("RF", model_rf), ("XGB", model_xgb)):
        for s_nama, sampler in sampler_dict.items():
            langkah = [("praproses", praproses())]
            if sampler is not None:
                langkah.append(("sampler", sampler))
            langkah.append(("model", buat()))
            try:
                hasil.append(evaluasi(skenario, model_nama, s_nama, Pipeline(langkah),
                                      X_train, y_tr, X_test, y_te))
            except Exception as e:                           # mis. ADASYN gagal pada data ini
                hasil.append({"skenario": skenario, "model": model_nama, "sampler": s_nama,
                              "status": f"gagal: {type(e).__name__}"})
                print("GAGAL", skenario, model_nama, s_nama, "->", str(e)[:90])
            print(skenario, model_nama, s_nama, "selesai", flush=True)
        # Pembanding tanpa sampling: pembobotan kelas
        if model_nama == "RF":
            hasil.append(evaluasi(skenario, "RF", "class_weight", Pipeline(
                [("praproses", praproses()), ("model", model_rf(class_weight="balanced"))]),
                X_train, y_tr, X_test, y_te))
        else:
            w = compute_sample_weight("balanced", y_tr)
            hasil.append(evaluasi(skenario, "XGB", "sample_weight", Pipeline(
                [("praproses", praproses()), ("model", model_xgb())]),
                X_train, y_tr, X_test, y_te, fit_kwargs={"model__sample_weight": w}))

# ---- Skenario BINER: normal (0) vs serangan (1); target "auto" menyamakan kedua kelas
hasil_biner = []
jalankan_semua("biner", yb_train, yb_test, konfigurasi(), hasil_biner)
tabel_biner = pd.DataFrame(hasil_biner)
tabel_biner.to_csv("output/hasil_biner.csv", index=False)
print(tabel_biner[["model", "sampler", "status", "f1_macro", "gmean", "waktu_fit_dtk"]].round(3).to_string(index=False))

## Kode 10.7 (kode107)

In [ ]:
# ---- Skenario MULTIKELAS: target sama dengan Bab 9 (BATAS dan TARGET = 2000 per kelas)
jumlah = Counter(y_train)
BATAS = 2000; TARGET = 2000                                    # CEK: sama dengan Bab 9
K = max(1, min(5, min(jumlah.values()) - 1))
target_rus = {k: min(v, BATAS) for k, v in jumlah.items()}
target_ros = {k: max(v, TARGET) for k, v in jumlah.items()}

hasil_multi = []
jalankan_semua("multikelas", y_train, y_test,
               konfigurasi(target_rus, target_ros, K), hasil_multi)
tabel_multi = pd.DataFrame(hasil_multi)
tabel_multi.to_csv("output/hasil_multikelas.csv", index=False)
rk = pd.DataFrame(recall_kelas)
rk["kelas_nama"] = rk.apply(lambda r: nama_kelas[r["kelas"]] if r["skenario"] == "multikelas"
                            else ["normal", "serangan"][r["kelas"]], axis=1)
rk.to_csv("output/recall_per_kelas.csv", index=False)
print(tabel_multi[["model", "sampler", "status", "f1_macro", "gmean", "waktu_fit_dtk"]].round(3).to_string(index=False))

## Kode 10.8 (kode108)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

urutan = ["Baseline", "RUS", "ROS", "SMOTE", "Borderline-SMOTE", "ADASYN",
          "SMOTE-Tomek", "SMOTE-ENN", "class_weight", "sample_weight"]
warna_model = (("RF", "#2E9C93", -0.14), ("XGB", "#E07B39", 0.14))
y = np.arange(len(urutan))

# Grafik titik (bukan batang): sumbu dipotong agar selisih kecil terlihat; garis putus = skor Baseline
fig, ax = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)
for a, (nama, t) in zip(ax, (("Klasifikasi biner", tabel_biner), ("Klasifikasi multikelas", tabel_multi))):
    for model, warna, dy in warna_model:
        d = t[(t["status"] == "ok") & (t["model"] == model)].set_index("sampler").reindex(urutan)
        a.scatter(d["f1_macro"], y + dy, color=warna, s=40, label=model, zorder=3)
        a.axvline(d.loc["Baseline", "f1_macro"], color=warna, ls="--", lw=0.9, alpha=0.7)
    a.set_title(nama); a.set_xlabel("F1 macro pada data uji (sumbu dipotong)")
    a.set_yticks(y); a.set_yticklabels(urutan); a.grid(axis="y", alpha=0.25)
ax[0].invert_yaxis()
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles[:2], labels[:2], loc="lower center", ncol=2, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig("output/g_f1_macro.png", dpi=200)
plt.show()

## Kode 10.9 (kode109c)

In [ ]:
# Mengapa F1 macro berubah? Hitung prediksi dan prediksi benar untuk tiga kelas terkecil (multikelas)
langka = ["Metasploit_Brute_Force_SSH", "NMAP_FIN_SCAN", "Wipro_bulb"]
baris = []
for model in ("RF", "XGB"):
    for sampler in ("Baseline", "RUS", "SMOTE", "Borderline-SMOTE", "SMOTE-ENN"):
        yp = np.load(f"output/prediksi/multikelas_{model}_{sampler}.npz")["y_pred"]
        for nama in langka:
            k = nama_kelas.index(nama)
            baris.append({"model": model, "sampler": sampler, "kelas": nama,
                          "sampel_uji": int((y_test == k).sum()),
                          "diprediksi": int((yp == k).sum()),
                          "benar": int(((yp == k) & (y_test == k)).sum())})
langka_df = pd.DataFrame(baris)
langka_df["presisi"] = (langka_df["benar"] / langka_df["diprediksi"]).round(2)
langka_df["recall"] = (langka_df["benar"] / langka_df["sampel_uji"]).round(2)
langka_df.to_csv("output/kelas_langka_prediksi.csv", index=False)

langka_df["benar/diprediksi"] = langka_df["benar"].astype(str) + "/" + langka_df["diprediksi"].astype(str)
for model in ("RF", "XGB"):
    p = langka_df[langka_df["model"] == model].pivot(index="kelas", columns="sampler", values="benar/diprediksi")
    print(model, "(benar/diprediksi; sampel uji: Metasploit 11, FIN 8, Wipro 66)")
    print(p[["Baseline", "RUS", "SMOTE", "Borderline-SMOTE", "SMOTE-ENN"]].to_string())

## Kode 10.10 (kode109)

In [ ]:
# Uji kepekaan: bagaimana bila kolom kategorikal (proto, service) ikut? Hanya Random Forest, multikelas.
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTENC

def praproses_kat():
    return ColumnTransformer([
        ("num", SkPipeline([("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler())]), kolom_num),
        ("kat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), kolom_kat)])

# Setelah ColumnTransformer, kolom kategorikal berada di posisi paling kanan
idx_kat = list(range(len(kolom_num), len(kolom_num) + len(kolom_kat)))
smote_kat = dict(sampling_strategy=target_ros, k_neighbors=K, random_state=RANDOM_STATE)

varian = {
    "numerik saja (Bab 10.7)": None,
    "kat. Baseline": [],
    "kat. SMOTE (kategori dianggap angka)": [("sampler", SMOTE(**smote_kat))],
    "kat. SMOTENC": [("sampler", SMOTENC(categorical_features=idx_kat, **smote_kat))],
}
hasil_kat = []
for nama, langkah in varian.items():
    if langkah is None:                                   # salin hasil numerik-saja dari tabel_multi
        r = tabel_multi[(tabel_multi.model == "RF") & (tabel_multi.sampler == "SMOTE")].iloc[0].to_dict()
        r["sampler"] = nama + " SMOTE"; hasil_kat.append(r); continue
    pipa = Pipeline([("praproses", praproses_kat())] + langkah +
                    [("model", model_rf())])
    hasil_kat.append(evaluasi("multikelas", "RF", nama, pipa, X_train, y_train, X_test, y_test))
    print(nama, "selesai", flush=True)
tabel_kat = pd.DataFrame(hasil_kat)
tabel_kat.to_csv("output/hasil_kepekaan_kategorikal.csv", index=False)
print(tabel_kat[["sampler", "f1_macro", "gmean", "waktu_fit_dtk"]].round(3).to_string(index=False))

## Kode 10.11 (kode109b)

In [ ]:
# Uji kepekaan kedua: apakah nomor port membantu model "menghafal sesi"? Multikelas, tanpa sampler.
port = ["id.orig_p", "id.resp_p"]
varian_port = {"port semua fitur": kolom_num,
               "port tanpa orig_p": [c for c in kolom_num if c != "id.orig_p"],
               "port tanpa kedua": [c for c in kolom_num if c not in port]}
hasil_port = []
for nama, kolom in varian_port.items():
    pra = ColumnTransformer([("num", SkPipeline([("imputer", SimpleImputer(strategy="median")),
                                                 ("scaler", StandardScaler())]), kolom)], remainder="drop")
    for model_nama, buat in (("RF", model_rf), ("XGB", model_xgb)):
        hasil_port.append(evaluasi("multikelas", model_nama, nama,
                                   Pipeline([("praproses", pra), ("model", buat())]),
                                   X_train, y_train, X_test, y_test))
        print(nama, model_nama, "selesai", flush=True)
tabel_port = pd.DataFrame(hasil_port)
tabel_port["jumlah_fitur"] = tabel_port["sampler"].map({n: len(k) for n, k in varian_port.items()})
tabel_port.to_csv("output/hasil_kepekaan_port.csv", index=False)
print(tabel_port[["model", "sampler", "jumlah_fitur", "f1_macro", "gmean"]].round(4).to_string(index=False))

## Kode 10.12 (kode1010)

In [ ]:
# Waktu latih (fit; termasuk praproses dan sampling) dan waktu prediksi seluruh data uji
print("CPU:", os.cpu_count(), "inti |", platform.platform())
gabung = pd.concat([tabel_biner, tabel_multi])
gabung = gabung[gabung["status"] == "ok"].copy()
gabung["ms_per_1000_baris"] = gabung["waktu_prediksi_dtk"] / len(X_test) * 1000 * 1000
waktu = gabung[["skenario", "model", "sampler", "waktu_fit_dtk", "waktu_prediksi_dtk", "ms_per_1000_baris"]]
waktu.to_csv("output/waktu.csv", index=False)
print(waktu.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
d = gabung[gabung["skenario"] == "multikelas"]
for a, kolom, judul, tick in ((ax[0], "waktu_fit_dtk", "Waktu latih (detik, skala log)", [1, 2, 5, 10, 20, 50]),
                              (ax[1], "ms_per_1000_baris", "Waktu prediksi (ms per 1.000 baris, skala log)", [3, 4, 5, 6, 8, 10])):
    for model, warna, dy in warna_model:
        s = d[d["model"] == model].set_index("sampler").reindex(urutan)
        a.scatter(s[kolom], y + dy, color=warna, s=40, label=model, zorder=3)
    a.set_xscale("log"); a.set_xticks(tick); a.xaxis.set_major_formatter(ScalarFormatter())
    a.minorticks_off(); a.set_title(judul, fontsize=10); a.grid(axis="y", alpha=0.25)
ax[0].set_yticks(y); ax[0].set_yticklabels(urutan); ax[0].invert_yaxis()
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles[:2], labels[:2], loc="lower center", ncol=2, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig("output/g_waktu.png", dpi=200)
plt.show()

## Kode 10.13 (kode1011)

In [ ]:
# Dokumentasi eksperimen: konfigurasi, versi, spesifikasi mesin, dan log gabungan
konfigurasi_eksp = {
    "versi": {lib.__name__: lib.__version__ for lib in (np, pd, sklearn, imblearn, xgboost)},
    "random_state": RANDOM_STATE,
    "mesin": {"cpu_inti": os.cpu_count(), "platform": platform.platform()},
    "fitur": f"{len(kolom_num)} numerik berskala; kategorikal tidak dipakai",
    "model": {"RF": model_rf().get_params(), "XGB": {k: v for k, v in model_xgb().get_params().items()
                                                       if k in ("n_estimators", "max_depth", "learning_rate", "tree_method", "random_state")}},
    "sampler_multikelas": {"BATAS": BATAS, "TARGET": TARGET, "k_neighbors": int(K)},
    "sampler_biner": "sampling_strategy=auto (kedua kelas disamakan)",
    "evaluasi": "latih pada X_train; nilai pada X_test asli; waktu prediksi = median 5 kali",
}
with open("output/konfigurasi_bab10.json", "w") as f:
    json.dump(konfigurasi_eksp, f, indent=2, ensure_ascii=False, default=str)

log = pd.concat([tabel_biner, tabel_multi, tabel_kat]).reset_index(drop=True)
log.to_csv("output/log_eksperimen_bab10.csv", index=False)
print("Konfigurasi dan log tersimpan; jumlah baris log:", len(log))
print("Konfigurasi gagal:", log[log["status"] != "ok"][["skenario", "model", "sampler", "status"]].values.tolist())